# CL-VSSC v3: Closed-Loop Feedback-Regulated Variable Separation & Controllable Spectral Clustering

Self-contained GPU notebook covering 10-TCGA cancer cohorts ($N=3{,}113$), Smooth PI vs. DZ-PI/P-only/open-loop/Null-SNR ablations, partial multi-omics, three-modality fault injection, NeurIPS GSE194122 CITE-seq ($90{,}261$ BMMCs, Global & Double-Centered Anchor-Local PI), and control sensitivity sweeps.


In [ ]:
# Cell 1: Check GPU availability, mount Google Drive, configure categorized output & weight directories
import datetime, json, shutil, subprocess, sys
from pathlib import Path
import numpy as np
import torch

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    subprocess.run(["nvidia-smi", "-L"], check=False)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lifelines", "anndata", "h5py", "requests"])

OUT_DIR = Path("./results_v2")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR_V3 = Path("./results_v3")
OUT_DIR_V3.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR = Path("./weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("./tcga_benchmark_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
SC_DIR = Path("./sc_data")
SC_DIR.mkdir(parents=True, exist_ok=True)

# Auto-mount Google Drive when running inside Google Colab
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/Paper/Paper2_CL-VSSC_ClosedLoop_Control")
else:
    DRIVE_ROOT = Path("./drive_backup/Paper/Paper2_CL-VSSC_ClosedLoop_Control")

DRIVE_CATEGORIES = {
    "results_v2": DRIVE_ROOT / "results_v2",
    "results_v3": DRIVE_ROOT / "results_v3",
    "weights": DRIVE_ROOT / "weights",
    "bundles": DRIVE_ROOT / "bundles",
    "logs": DRIVE_ROOT / "logs",
}
for _cat_dir in DRIVE_CATEGORIES.values():
    _cat_dir.mkdir(parents=True, exist_ok=True)


def sync_to_drive(local_path: Path, category: str) -> Path:
    """Copy a file or directory into the categorized Google Drive folder."""
    local_path = Path(local_path)
    if category not in DRIVE_CATEGORIES:
        raise ValueError(f"Unknown Drive category: {category}")
    dest_dir = DRIVE_CATEGORIES[category]
    if local_path.is_dir():
        copied = 0
        for item in sorted(local_path.iterdir()):
            if item.is_file():
                shutil.copy2(item, dest_dir / item.name)
                copied += 1
        print(f"[Drive Sync] {local_path}/ ({copied} files) -> {dest_dir}")
        return dest_dir
    elif local_path.is_file():
        dst = dest_dir / local_path.name
        shutil.copy2(local_path, dst)
        print(f"[Drive Sync] {local_path.name} -> {dst}")
        return dst
    return dest_dir


def save_run_weights(run_name: str, res_dict: dict) -> Path:
    """Persist learned modal weights, embeddings, labels, and control histories as .npz."""
    payload = {}
    for key in ["final_weights", "labels", "F", "weight_history", "spectral_gap_history", "V_history"]:
        if key in res_dict and res_dict[key] is not None:
            payload[key] = np.asarray(res_dict[key])
    out_file = WEIGHTS_DIR / f"{run_name}.npz"
    np.savez_compressed(out_file, **payload)
    sync_to_drive(out_file, "weights")
    return out_file

print("Drive archive root:", DRIVE_ROOT)


In [ ]:
# Cell 2: Self-contained CL-VSSC v2 (Dense + Anchor + Partial Masks), Deep/Graph Baselines, and TCGA Loader
"""Closed-Loop Feedback-Regulated Variable Separation and Controllable Spectral Clustering (CL-VSSC v2)."""
from __future__ import annotations

from typing import Dict, List, Optional, Tuple
import numpy as np
import torch
from sklearn.cluster import KMeans, MiniBatchKMeans


def get_device(device: Optional[str] = None) -> torch.device:
    if device is not None:
        return torch.device(device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def preprocess_views(
    views: List[np.ndarray],
    device: torch.device,
    masks: Optional[List[np.ndarray]] = None,
) -> Tuple[List[torch.Tensor], List[torch.Tensor]]:
    """Column-standardize each view (using observed rows if masks supplied) and normalize by sqrt(d_v)."""
    processed: List[torch.Tensor] = []
    mask_tensors: List[torch.Tensor] = []
    for v_idx, v in enumerate(views):
        arr = np.asarray(v, dtype=np.float32)
        n = arr.shape[0]
        if masks is not None:
            m = np.asarray(masks[v_idx], dtype=bool)
            obs = arr[m] if m.any() else arr
            mean = np.nanmean(obs, axis=0, keepdims=True)
            std = np.nanstd(obs, axis=0, keepdims=True)
            mean = np.nan_to_num(mean, nan=0.0)
            std = np.where((~np.isfinite(std)) | (std < 1e-6), 1.0, std)
            arr_clean = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            normed = (arr_clean - mean) / std
            normed[~m] = 0.0
            scale = np.sqrt(max(1, normed.shape[1])) * np.sqrt(max(1, int(m.sum())) / max(1, n))
            t = torch.tensor(normed / scale, dtype=torch.float32, device=device)
            mt = torch.tensor(m.astype(np.float32), dtype=torch.float32, device=device)
        else:
            arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            mean = arr.mean(axis=0, keepdims=True)
            std = arr.std(axis=0, keepdims=True)
            std = np.where(std < 1e-6, 1.0, std)
            normed = (arr - mean) / std
            scale = np.sqrt(max(1, normed.shape[1]))
            t = torch.tensor(normed / scale, dtype=torch.float32, device=device)
            mt = torch.ones((n,), dtype=torch.float32, device=device)
        processed.append(t)
        mask_tensors.append(mt)
    return processed, mask_tensors


def build_knn_affinity(X: torch.Tensor, k: int, obs_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Self-tuning k-NN Gaussian affinity with symmetric row normalization (supports partial observation mask)."""
    n = X.shape[0]
    if obs_mask is not None and float(obs_mask.min().item()) < 0.5:
        obs_idx = torch.nonzero(obs_mask > 0.5, as_tuple=False).squeeze(1)
        A_full = torch.zeros((n, n), dtype=X.dtype, device=X.device)
        if len(obs_idx) < 3:
            return A_full
        A_sub = build_knn_affinity(X[obs_idx], k=min(k, len(obs_idx) - 1), obs_mask=None)
        A_full[obs_idx[:, None], obs_idx[None, :]] = A_sub
        return A_full

    k_eff = max(2, min(k, n - 1))
    dist2 = torch.cdist(X, X, p=2.0).pow(2)
    dist2.fill_diagonal_(float("inf"))
    knn_vals, knn_idx = torch.topk(dist2, k=k_eff, dim=1, largest=False)
    sig2 = knn_vals.mean(dim=1).clamp_min(1e-6)

    dist2.fill_diagonal_(0.0)
    denom = sig2[:, None] + sig2[None, :] + 1e-6
    kernel = torch.exp(-dist2 / denom)

    mask = torch.zeros((n, n), dtype=torch.bool, device=X.device)
    row_idx = torch.arange(n, device=X.device)[:, None]
    mask[row_idx, knn_idx] = True
    mask = mask | mask.T
    mask.fill_diagonal_(False)

    A = kernel * mask.float()
    row_sum = A.sum(dim=1, keepdim=True).clamp_min(1e-8)
    P = A / row_sum
    return 0.5 * (P + P.T)


def normalized_laplacian(A: torch.Tensor) -> torch.Tensor:
    """Compute symmetric normalized graph Laplacian L(A) = I - D^{-1/2} A D^{-1/2}."""
    n = A.shape[0]
    deg = A.sum(dim=1).clamp_min(1e-8)
    d_inv_sqrt = deg.rsqrt()
    A_norm = d_inv_sqrt[:, None] * A * d_inv_sqrt[None, :]
    L = torch.eye(n, dtype=A.dtype, device=A.device) - 0.5 * (A_norm + A_norm.T)
    return L


def doubly_stochastic_normalize(A: torch.Tensor, n_iters: int = 5) -> torch.Tensor:
    """Symmetric Sinkhorn-like scaling onto balanced affinity scale."""
    M = A.clamp_min(0.0)
    M = 0.5 * (M + M.T)
    eps = 1e-8
    for _ in range(n_iters):
        row_sum = M.sum(dim=1, keepdim=True).clamp_min(eps)
        M = M / row_sum
        M = 0.5 * (M + M.T)
    row_sum = M.sum(dim=1, keepdim=True).clamp_min(eps)
    M = M / row_sum
    return 0.5 * (M + M.T)


def polar_orthogonalize(M: torch.Tensor) -> torch.Tensor:
    """Nearest column-orthonormal matrix on the Stiefel manifold via thin SVD."""
    U, _, Vt = torch.linalg.svd(M, full_matrices=False)
    return U @ Vt


def fast_left_svd(M: torch.Tensor, rank: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute top-rank left singular vectors and singular values efficiently."""
    min_dim = min(M.shape)
    q = min(rank, min_dim)
    if min_dim <= 640 or q >= min_dim // 2:
        U, S_vals, _ = torch.linalg.svd(M, full_matrices=False)
        return U[:, :q].clone(), S_vals[:q].clone()
    over = min(min_dim, q + 8)
    U, S_vals, _ = torch.svd_lowrank(M, q=over, niter=3)
    return U[:, :q].clone(), S_vals[:q].clone()


def spectral_eigendecomposition(
    S: torch.Tensor, n_clusters: int
) -> Tuple[torch.Tensor, torch.Tensor, float, float]:
    """Compute bottom-(c+1) normalized Laplacian eigenvalues and bottom-c eigenvectors."""
    deg = S.sum(dim=1).clamp_min(1e-8)
    d_inv_sqrt = deg.rsqrt()
    S_norm = d_inv_sqrt[:, None] * S * d_inv_sqrt[None, :]
    S_norm = 0.5 * (S_norm + S_norm.T)
    evals_s, evecs_s = torch.linalg.eigh(S_norm)
    lap_evals = (1.0 - evals_s.flip(0)).clamp_min(0.0)
    F = evecs_s[:, -n_clusters:].flip(1)
    spec_trace = float(lap_evals[:n_clusters].sum().item())
    gap = float((lap_evals[n_clusters] - lap_evals[n_clusters - 1]).item()) if len(lap_evals) > n_clusters else 0.0
    return F, d_inv_sqrt, spec_trace, gap


def _shuffle_observed_columns(
    X: torch.Tensor, obs_mask: Optional[torch.Tensor], seed: int
) -> torch.Tensor:
    """Independently permute observed entries within each feature column to form a structureless null baseline."""
    gen = torch.Generator(device=X.device)
    gen.manual_seed(seed)
    X_null = X.clone()
    if obs_mask is not None and float(obs_mask.min().item()) < 0.5:
        obs_idx = torch.nonzero(obs_mask > 0.5, as_tuple=False).squeeze(1)
        if len(obs_idx) < 4:
            return X_null
        x_obs = X[obs_idx]
        rand_keys = torch.rand((len(obs_idx), X.shape[1]), generator=gen, device=X.device)
        perm = torch.argsort(rand_keys, dim=0)
        X_null[obs_idx] = torch.gather(x_obs, 0, perm)
        return X_null
    rand_keys = torch.rand(X.shape, generator=gen, device=X.device)
    perm = torch.argsort(rand_keys, dim=0)
    return torch.gather(X, 0, perm)


def _compute_dense_null_cut(
    X: torch.Tensor, obs_mask: Optional[torch.Tensor], k_nn: int, n_clusters: int, seed: int
) -> float:
    """Null-hypothesis intrinsic normalized cut after independent column permutation."""
    x_null = _shuffle_observed_columns(X, obs_mask, seed=seed)
    if obs_mask is not None and float(obs_mask.min().item()) < 0.5:
        obs_idx = torch.nonzero(obs_mask > 0.5, as_tuple=False).squeeze(1)
        if len(obs_idx) < 4:
            return 1.0
        x_sub = x_null[obs_idx]
        A_null = build_knn_affinity(x_sub, k=min(k_nn, len(obs_idx) - 1), obs_mask=None)
        c_eff = min(n_clusters, max(2, len(obs_idx) - 1))
    else:
        A_null = build_knn_affinity(x_null, k=k_nn, obs_mask=None)
        c_eff = n_clusters
    _, _, null_tr, _ = spectral_eigendecomposition(A_null, c_eff)
    return max(0.15, float(null_tr) / max(1, n_clusters))


def _compute_anchor_null_cut(
    X: torch.Tensor, k_anchor: int, n_anchors: int, n_clusters: int, seed: int
) -> float:
    """Fast subsampled null bipartite normalized cut after independent column permutation."""
    gen = torch.Generator(device=X.device)
    gen.manual_seed(seed + 500)
    sub_n = min(X.shape[0], 4096)
    if sub_n < X.shape[0]:
        sub_idx = torch.randperm(X.shape[0], generator=gen, device=X.device)[:sub_n]
        x_sub = X[sub_idx]
    else:
        x_sub = X
    x_null = _shuffle_observed_columns(x_sub, None, seed=seed)
    m_sub = min(n_anchors, max(n_clusters + 4, sub_n // 4))
    anc_idx = torch.randperm(sub_n, generator=gen, device=X.device)[:m_sub]
    B_null, _ = build_bipartite_affinity(x_null, x_null[anc_idx], k_anchor=min(k_anchor, m_sub))
    _, _, _, null_tr, _ = bipartite_spectral_eigendecomposition(B_null, n_clusters)
    return max(0.15, float(null_tr) / max(1, n_clusters))


def _apply_deadzone_pi(
    e_vec: torch.Tensor,
    I_err: torch.Tensor,
    w: torch.Tensor,
    dead_zone: float,
    beta_i: float,
    kappa_i: float,
    eta_w: float,
    tau: float,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Dead-Zone PI mirror-descent simplex update supporting 1D (V,) and 2D (m, V) weight matrices."""
    if dead_zone > 0.0:
        d_vec = torch.clamp(e_vec - e_vec.min(dim=-1, keepdim=True).values - dead_zone, min=0.0)
    else:
        d_vec = e_vec
    I_next = (1.0 - beta_i) * I_err + beta_i * d_vec
    signal = d_vec + kappa_i * I_next
    signal_centered = signal - signal.mean(dim=-1, keepdim=True)
    log_w = (1.0 - eta_w) * torch.log(w.clamp_min(1e-12)) - (eta_w / tau) * signal_centered
    w_next = torch.softmax(log_w, dim=-1)
    return w_next, I_next


class CLVSSC:
    """Closed-Loop PI Feedback Variable Separation & Controllable Spectral Clustering."""

    def __init__(
        self,
        n_clusters: int,
        shared_dim: int = 24,
        spec_dim: int = 6,
        knn_k: Optional[int] = None,
        max_iter: int = 45,
        warmup_iters: int = 15,
        alpha_graph: float = 1.0,
        gamma_rec: float = 0.30,
        gamma_cons: float = 0.08,
        lambda_z: float = 2.0,
        beta_u: float = 0.35,
        eta_z: float = 0.35,
        eta_s: float = 0.35,
        rho_a: float = 0.25,
        eta_w: float = 0.30,
        beta_i: float = 0.4,
        kappa_i: float = 0.35,
        tau: float = 0.20,
        dead_zone: float = 0.0,
        mu_init: float = 0.0,
        eta_mu: float = 0.05,
        mu_max: float = 0.12,
        eps_spec: float = 0.02,
        use_feedback: bool = True,
        use_spectral_control: bool = True,
        use_variable_separation: bool = True,
        use_null_snr: bool = False,
        seed: int = 42,
        device: Optional[str] = None,
    ):
        self.n_clusters = n_clusters
        self.shared_dim = shared_dim
        self.spec_dim = spec_dim
        self.knn_k = knn_k
        self.max_iter = max_iter
        self.warmup_iters = warmup_iters
        self.alpha_graph = alpha_graph
        self.gamma_rec = gamma_rec
        self.gamma_cons = gamma_cons
        self.lambda_z = lambda_z
        self.beta_u = beta_u
        self.eta_z = eta_z
        self.eta_s = eta_s
        self.rho_a = rho_a
        self.eta_w = eta_w
        self.beta_i = beta_i
        self.kappa_i = kappa_i
        self.tau = tau
        self.dead_zone = dead_zone
        self.mu_init = mu_init
        self.eta_mu = eta_mu
        self.mu_max = mu_max
        self.eps_spec = eps_spec
        self.use_feedback = use_feedback
        self.use_spectral_control = use_spectral_control
        self.use_variable_separation = use_variable_separation
        self.use_null_snr = use_null_snr
        self.seed = seed
        self.device = get_device(device)

    @torch.no_grad()
    def fit(self, views: List[np.ndarray], masks: Optional[List[np.ndarray]] = None) -> Dict[str, object]:
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)

        has_missing = masks is not None
        X_list, M_list = preprocess_views(views, self.device, masks=masks)
        pair_masks = [m[:, None] * m[None, :] for m in M_list]
        obs_counts = [int((m > 0.5).sum().item()) for m in M_list]

        V = len(X_list)
        n = X_list[0].shape[0]
        k_nn = self.knn_k if self.knn_k is not None else max(8, min(25, n // 6))
        x_norms = [float((x * m[:, None]).pow(2).sum().item()) + 1e-8 for x, m in zip(X_list, M_list)]
        op_norm2 = [float(fast_left_svd(x, 2)[1][0].pow(2).item()) + 1e-8 for x in X_list]

        # Initialize shared state Z via SVD of concatenated views
        X_cat = torch.cat(X_list, dim=1)
        r = min(self.shared_dim, n - 1, X_cat.shape[1])
        Z, _ = fast_left_svd(X_cat, r)

        # Iterative subspace imputation initialization on unobserved rows
        X_cur: List[torch.Tensor] = []
        P_list: List[torch.Tensor] = []
        for v_idx, x in enumerate(X_list):
            p_v = Z.T @ x
            if has_missing:
                m_col = M_list[v_idx][:, None]
                x_filled = m_col * x + (1.0 - m_col) * (Z @ p_v)
                p_v = Z.T @ x_filled
                X_cur.append(x_filled)
            else:
                X_cur.append(x)
            P_list.append(p_v)

        # Initialize orthogonal modality-specific factors U^(v), Q^(v)
        U_list: List[Optional[torch.Tensor]] = []
        Q_list: List[Optional[torch.Tensor]] = []
        for v_idx, x_f in enumerate(X_cur):
            if self.use_variable_separation:
                r_v = min(self.spec_dim, n - r - 1 if n - r > 2 else max(1, n // 6), x_f.shape[1])
                r_v = max(1, min(r_v, x_f.shape[1]))
                res = x_f - Z @ P_list[v_idx]
                res = res - Z @ (Z.T @ res)
                u_v, _ = fast_left_svd(res, r_v)
                q_v = u_v.T @ res
                U_list.append(u_v)
                Q_list.append(q_v)
            else:
                U_list.append(None)
                Q_list.append(None)

        dof_scales = [
            max(
                0.25,
                1.0
                - (r + (U_list[v_i].shape[1] if U_list[v_i] is not None else 0))
                / (min(obs_counts[v_i], X_list[v_i].shape[1]) + r + (U_list[v_i].shape[1] if U_list[v_i] is not None else 0)),
            )
            for v_i in range(V)
        ]

        # Initialize per-view k-NN affinities on softly denoised projections
        A_list: List[torch.Tensor] = []
        L_list: List[torch.Tensor] = []
        own_cuts: List[float] = []
        null_cuts: List[float] = []
        for v_idx, x_f in enumerate(X_cur):
            x_sh = x_f - self.beta_u * (U_list[v_idx] @ Q_list[v_idx]) if self.use_variable_separation else x_f
            m_v = M_list[v_idx] if has_missing else None
            A_v = build_knn_affinity(x_sh, k=k_nn, obs_mask=m_v)
            A_list.append(A_v)
            if has_missing:
                obs_idx = torch.nonzero(M_list[v_idx] > 0.5, as_tuple=False).squeeze(1)
                A_sub = A_v[obs_idx[:, None], obs_idx[None, :]]
                L_sub = normalized_laplacian(A_sub)
                L_full = torch.zeros_like(A_v)
                L_full[obs_idx[:, None], obs_idx[None, :]] = L_sub
                L_list.append(L_full)
                _, _, own_tr, _ = spectral_eigendecomposition(A_sub, min(self.n_clusters, max(2, len(obs_idx) - 1)))
                raw_cut = own_tr / max(1, self.n_clusters)
            else:
                L_list.append(normalized_laplacian(A_v))
                _, _, own_tr, _ = spectral_eigendecomposition(A_v, self.n_clusters)
                raw_cut = own_tr / max(1, self.n_clusters)
            if self.use_null_snr:
                c_n = _compute_dense_null_cut(x_sh, m_v, k_nn=k_nn, n_clusters=self.n_clusters, seed=self.seed + 1000 + v_idx)
                null_cuts.append(c_n)
                own_cuts.append(raw_cut / c_n)
            else:
                null_cuts.append(1.0)
                own_cuts.append(raw_cut)

        w = torch.full((V,), 1.0 / V, dtype=torch.float32, device=self.device)
        I_err = torch.zeros((V,), dtype=torch.float32, device=self.device)

        def fuse_weighted_affinities(cur_w: torch.Tensor) -> torch.Tensor:
            if not has_missing:
                S_w = sum(cur_w[v] * A_list[v] for v in range(V))
                return S_w
            num = sum(cur_w[v] * pair_masks[v] * A_list[v] for v in range(V))
            den = sum(cur_w[v] * pair_masks[v] for v in range(V))
            S_w = torch.where(den > 1e-8, num / den.clamp_min(1e-8), torch.zeros_like(num))
            S_w.fill_diagonal_(0.0)
            row_s = S_w.sum(dim=1, keepdim=True)
            if float(row_s.min().item()) < 1e-6:
                Z_aff = build_knn_affinity(Z, k=min(8, n - 1))
                S_w = torch.where(row_s > 1e-6, S_w, Z_aff)
            return 0.5 * (S_w + S_w.T)

        S = fuse_weighted_affinities(w)
        S = 0.5 * (S + S.T)

        mu = self.mu_init if self.use_spectral_control else 0.0
        F, d_inv_sqrt, spec_trace, gap = spectral_eigendecomposition(S, self.n_clusters)

        weights_history: List[List[float]] = []
        lyapunov_history: List[float] = []
        spectral_gap_history: List[float] = []
        spectral_trace_history: List[float] = []
        mu_history: List[float] = []

        def compute_errors_and_lyapunov(
            cur_Z: torch.Tensor,
            cur_P: List[torch.Tensor],
            cur_U: List[Optional[torch.Tensor]],
            cur_Q: List[Optional[torch.Tensor]],
            cur_L: List[torch.Tensor],
            cur_own_cuts: List[float],
            cur_F: torch.Tensor,
            cur_w: torch.Tensor,
            cur_trace: float,
        ) -> Tuple[torch.Tensor, float]:
            errs = []
            lyap = 0.0
            for v_i, x in enumerate(X_list):
                recon = cur_Z @ cur_P[v_i]
                if cur_U[v_i] is not None and cur_Q[v_i] is not None:
                    recon = recon + cur_U[v_i] @ cur_Q[v_i]
                diff = (x - recon) * M_list[v_i][:, None]
                rec_rel = (float(diff.pow(2).sum().item()) / x_norms[v_i]) / dof_scales[v_i]
                cons_raw = float(torch.trace(cur_F.T @ cur_L[v_i] @ cur_F).item())
                if has_missing:
                    obs_frac = max(0.1, obs_counts[v_i] / max(1, n))
                    cons_cut = (cons_raw / obs_frac) / max(1, self.n_clusters)
                else:
                    cons_cut = cons_raw / max(1, self.n_clusters)
                own_cut = cur_own_cuts[v_i]
                e_v = own_cut + self.gamma_cons * cons_cut + self.gamma_rec * rec_rel
                errs.append(e_v)
                lyap += float(cur_w[v_i].item()) * (rec_rel + self.alpha_graph * (own_cut + self.gamma_cons * cons_cut))
            entropy_term = float((cur_w * torch.log(cur_w.clamp_min(1e-12))).sum().item())
            spec_weight = self.mu_max if self.use_spectral_control else 0.0
            lyap += spec_weight * cur_trace + self.tau * entropy_term
            return torch.tensor(errs, dtype=torch.float32, device=self.device), lyap

        _, V_t = compute_errors_and_lyapunov(Z, P_list, U_list, Q_list, L_list, own_cuts, F, w, spec_trace)
        weights_history.append(w.detach().cpu().tolist())
        lyapunov_history.append(V_t)
        spectral_gap_history.append(gap)
        spectral_trace_history.append(spec_trace)
        mu_history.append(mu)

        for t in range(self.max_iter):
            # Step A: Orthogonal Variable Separation update (Z, P, U, Q)
            deg_s = S.sum(dim=1).clamp_min(1e-8)
            d_inv = deg_s.rsqrt()
            S_norm = d_inv[:, None] * S * d_inv[None, :]
            grad_Z = self.lambda_z * (S_norm @ Z)
            for v_i, x in enumerate(X_list):
                if has_missing:
                    m_col = M_list[v_i][:, None]
                    X_cur[v_i] = m_col * x + (1.0 - m_col) * (Z @ P_list[v_i])
                x_sh = X_cur[v_i] - (U_list[v_i] @ Q_list[v_i]) if self.use_variable_separation else X_cur[v_i]
                P_list[v_i] = Z.T @ x_sh
                grad_Z = grad_Z + (float(w[v_i].item()) / op_norm2[v_i]) * (x_sh @ P_list[v_i].T)

            Z_target = polar_orthogonalize(grad_Z)
            Z = polar_orthogonalize((1.0 - self.eta_z) * Z + self.eta_z * Z_target)

            for v_i in range(V):
                x_f = X_cur[v_i]
                if self.use_variable_separation:
                    res = x_f - Z @ (Z.T @ x_f)
                    r_v = U_list[v_i].shape[1]
                    U_cand, _ = fast_left_svd(res, r_v)
                    U_cand = U_cand - Z @ (Z.T @ U_cand)
                    U_cand = polar_orthogonalize(U_cand)
                    U_list[v_i] = U_cand
                    Q_list[v_i] = U_cand.T @ res
                    P_list[v_i] = Z.T @ (x_f - U_list[v_i] @ Q_list[v_i])
                else:
                    P_list[v_i] = Z.T @ x_f

            # Smoothly refresh per-view affinity graphs from softly denoised projection every 5 steps
            if self.use_variable_separation and (t + 1) % 5 == 0 and t < self.max_iter - 5:
                for v_i in range(V):
                    x_sh = X_cur[v_i] - self.beta_u * (U_list[v_i] @ Q_list[v_i])
                    m_v = M_list[v_i] if has_missing else None
                    A_new = build_knn_affinity(x_sh, k=k_nn, obs_mask=m_v)
                    A_v = (1.0 - self.rho_a) * A_list[v_i] + self.rho_a * A_new
                    A_list[v_i] = 0.5 * (A_v + A_v.T)
                    if has_missing:
                        obs_idx = torch.nonzero(M_list[v_i] > 0.5, as_tuple=False).squeeze(1)
                        A_sub = A_list[v_i][obs_idx[:, None], obs_idx[None, :]]
                        L_sub = normalized_laplacian(A_sub)
                        L_full = torch.zeros_like(A_list[v_i])
                        L_full[obs_idx[:, None], obs_idx[None, :]] = L_sub
                        L_list[v_i] = L_full
                        _, _, own_tr, _ = spectral_eigendecomposition(A_sub, min(self.n_clusters, max(2, len(obs_idx) - 1)))
                        own_cuts[v_i] = (own_tr / max(1, self.n_clusters)) / null_cuts[v_i]
                    else:
                        L_list[v_i] = normalized_laplacian(A_list[v_i])
                        _, _, own_tr, _ = spectral_eigendecomposition(A_list[v_i], self.n_clusters)
                        own_cuts[v_i] = (own_tr / max(1, self.n_clusters)) / null_cuts[v_i]

            # Step B: Consensus affinity S update + Dual spectral-gap feedback controller (after warmup)
            S_weighted = fuse_weighted_affinities(w)
            if self.use_spectral_control and t >= self.warmup_iters and mu > 0.0:
                f_norms = F.norm(dim=1, keepdim=True).clamp_min(1e-8)
                F_unit_t = F / f_norms
                H = torch.cdist(F_unit_t, F_unit_t, p=2.0).pow(2)
                S_target = (S_weighted - (0.14 * mu / k_nn) * H).clamp_min(0.0)
            else:
                S_target = S_weighted.clamp_min(0.0)
            S_target.fill_diagonal_(0.0)
            row_st = S_target.sum(dim=1, keepdim=True).clamp_min(1e-8)
            S_target = S_target / row_st
            S_target = 0.5 * (S_target + S_target.T)
            S = (1.0 - self.eta_s) * S + self.eta_s * S_target
            S = 0.5 * (S + S.T)

            F, d_inv_sqrt, spec_trace, gap = spectral_eigendecomposition(S, self.n_clusters)
            if self.use_spectral_control and t >= self.warmup_iters:
                mu = float(np.clip(mu + self.eta_mu * (spec_trace - self.eps_spec), 0.0, self.mu_max))

            # Step C: Closed-loop Dead-Zone PI / mirror-descent modality weight controller
            e_vec, _ = compute_errors_and_lyapunov(Z, P_list, U_list, Q_list, L_list, own_cuts, F, w, spec_trace)
            if self.use_feedback:
                w, I_err = _apply_deadzone_pi(
                    e_vec, I_err, w,
                    dead_zone=self.dead_zone,
                    beta_i=self.beta_i,
                    kappa_i=self.kappa_i,
                    eta_w=self.eta_w,
                    tau=self.tau,
                )

            _, V_t = compute_errors_and_lyapunov(Z, P_list, U_list, Q_list, L_list, own_cuts, F, w, spec_trace)
            weights_history.append(w.detach().cpu().tolist())
            lyapunov_history.append(V_t)
            spectral_gap_history.append(gap)
            spectral_trace_history.append(spec_trace)
            mu_history.append(mu)

        F_np = F.detach().cpu().numpy()
        row_norms = np.linalg.norm(F_np, axis=1, keepdims=True)
        row_norms = np.where(row_norms < 1e-12, 1.0, row_norms)
        F_unit = F_np / row_norms
        labels = KMeans(n_clusters=self.n_clusters, n_init=30, random_state=self.seed).fit_predict(F_unit)

        return {
            "labels": labels,
            "S": S.detach().cpu().numpy(),
            "Z": Z.detach().cpu().numpy(),
            "F": F_unit,
            "final_weights": w.detach().cpu().numpy(),
            "weights_history": np.array(weights_history),
            "lyapunov_history": np.array(lyapunov_history),
            "spectral_gap_history": np.array(spectral_gap_history),
            "spectral_trace_history": np.array(spectral_trace_history),
            "mu_history": np.array(mu_history),
        }


def build_bipartite_affinity(
    X: torch.Tensor, anchors: torch.Tensor, k_anchor: int = 15
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Row-normalized n x m bipartite affinity B and column-scaled B_tilde = B * diag(d_B^{-1/2})."""
    m = anchors.shape[0]
    k_eff = max(2, min(k_anchor, m))
    dist2 = torch.cdist(X, anchors, p=2.0).pow(2)
    knn_vals, knn_idx = torch.topk(dist2, k=k_eff, dim=1, largest=False)
    sig2 = knn_vals.mean(dim=1, keepdim=True).clamp_min(1e-6)
    weights = torch.exp(-knn_vals / (2.0 * sig2 + 1e-8))
    weights = weights / weights.sum(dim=1, keepdim=True).clamp_min(1e-8)

    B = torch.zeros((X.shape[0], m), dtype=X.dtype, device=X.device)
    row_ar = torch.arange(X.shape[0], device=X.device)[:, None]
    B[row_ar, knn_idx] = weights

    col_deg = B.sum(dim=0).clamp_min(1e-8)
    B_tilde = B * col_deg.rsqrt()[None, :]
    return B, B_tilde


def bipartite_spectral_eigendecomposition(
    B: torch.Tensor, n_clusters: int
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float, float]:
    """Exact O(n m^2 + m^3) normalized spectral decomposition of P = B D_B^{-1} B^T."""
    col_deg = B.sum(dim=0).clamp_min(1e-8)
    B_tilde = B * col_deg.rsqrt()[None, :]
    G = B_tilde.T @ B_tilde
    G = 0.5 * (G + G.T)
    evals, V_anc = torch.linalg.eigh(G)
    # eigh returns ascending; largest eigenvalues correspond to smallest Laplacian eigenvalues 1 - sigma^2
    sigma2 = evals.flip(0).clamp(min=1e-10, max=1.0)
    V_desc = V_anc.flip(1)

    c_eff = min(n_clusters, len(sigma2))
    s_top = sigma2[:c_eff].sqrt().clamp_min(1e-6)
    U_anc = V_desc[:, :c_eff]
    F = B_tilde @ (U_anc / s_top[None, :])

    lap_evals = (1.0 - sigma2).clamp_min(0.0)
    spec_trace = float(lap_evals[:c_eff].sum().item())
    gap = float((lap_evals[c_eff] - lap_evals[c_eff - 1]).item()) if len(lap_evals) > c_eff else 0.0
    return F, U_anc, B_tilde, spec_trace, gap


class CLVSSCAnchor:
    """Linear-time O(n m^2 + n d r) bipartite landmark closed-loop spectral controller for large cohorts."""

    def __init__(
        self,
        n_clusters: int,
        n_anchors: int = 384,
        anchor_k: int = 20,
        shared_dim: int = 24,
        spec_dim: int = 6,
        max_iter: int = 30,
        warmup_iters: int = 10,
        alpha_graph: float = 1.0,
        gamma_rec: float = 0.30,
        gamma_cons: float = 0.08,
        lambda_z: float = 2.0,
        beta_u: float = 0.35,
        eta_z: float = 0.35,
        eta_s: float = 0.35,
        rho_a: float = 0.25,
        eta_w: float = 0.30,
        beta_i: float = 0.4,
        kappa_i: float = 0.35,
        tau: float = 0.20,
        dead_zone: float = 0.0,
        mu_init: float = 0.0,
        eta_mu: float = 0.05,
        mu_max: float = 0.12,
        eps_spec: float = 0.02,
        use_feedback: bool = True,
        use_spectral_control: bool = True,
        use_variable_separation: bool = True,
        use_null_snr: bool = False,
        use_local_pi: bool = False,
        alpha_local: float = 0.25,
        seed: int = 42,
        device: Optional[str] = None,
    ):
        self.n_clusters = n_clusters
        self.n_anchors = n_anchors
        self.anchor_k = anchor_k
        self.shared_dim = shared_dim
        self.spec_dim = spec_dim
        self.max_iter = max_iter
        self.warmup_iters = warmup_iters
        self.alpha_graph = alpha_graph
        self.gamma_rec = gamma_rec
        self.gamma_cons = gamma_cons
        self.lambda_z = lambda_z
        self.beta_u = beta_u
        self.eta_z = eta_z
        self.eta_s = eta_s
        self.rho_a = rho_a
        self.eta_w = eta_w
        self.beta_i = beta_i
        self.kappa_i = kappa_i
        self.tau = tau
        self.dead_zone = dead_zone
        self.mu_init = mu_init
        self.eta_mu = eta_mu
        self.mu_max = mu_max
        self.eps_spec = eps_spec
        self.use_feedback = use_feedback
        self.use_spectral_control = use_spectral_control
        self.use_variable_separation = use_variable_separation
        self.use_null_snr = use_null_snr
        self.use_local_pi = use_local_pi
        self.alpha_local = alpha_local
        self.seed = seed
        self.device = get_device(device)

    @torch.no_grad()
    def fit(self, views: List[np.ndarray]) -> Dict[str, object]:
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)

        X_list, _ = preprocess_views(views, self.device)
        V = len(X_list)
        n = X_list[0].shape[0]
        m = min(self.n_anchors, max(self.n_clusters + 4, n // 4))
        k_a = min(self.anchor_k, m)
        x_norms = [float(x.pow(2).sum().item()) + 1e-8 for x in X_list]
        op_norm2 = [float(fast_left_svd(x, 2)[1][0].pow(2).item()) + 1e-8 for x in X_list]

        X_cat = torch.cat(X_list, dim=1)
        r = min(self.shared_dim, n - 1, X_cat.shape[1])
        Z, _ = fast_left_svd(X_cat, r)
        P_list = [Z.T @ x for x in X_list]

        U_list: List[Optional[torch.Tensor]] = []
        Q_list: List[Optional[torch.Tensor]] = []
        for v_idx, x in enumerate(X_list):
            if self.use_variable_separation:
                r_v = max(1, min(self.spec_dim, n - r - 1 if n - r > 2 else max(1, n // 6), x.shape[1]))
                res = x - Z @ P_list[v_idx]
                res = res - Z @ (Z.T @ res)
                u_v, _ = fast_left_svd(res, r_v)
                q_v = u_v.T @ res
                U_list.append(u_v)
                Q_list.append(q_v)
            else:
                U_list.append(None)
                Q_list.append(None)

        dof_scales = [
            max(
                0.25,
                1.0
                - (r + (U_list[v_i].shape[1] if U_list[v_i] is not None else 0))
                / (min(n, X_list[v_i].shape[1]) + r + (U_list[v_i].shape[1] if U_list[v_i] is not None else 0)),
            )
            for v_i in range(V)
        ]

        # Select representative landmark cells via fast MiniBatchKMeans centroids mapped to nearest real cells
        z_np = Z.detach().cpu().numpy()
        km = MiniBatchKMeans(n_clusters=m, n_init=3, batch_size=min(2048, n), max_iter=25, random_state=self.seed)
        km.fit(z_np)
        centers = torch.tensor(km.cluster_centers_, dtype=torch.float32, device=self.device)
        d2_anc = torch.cdist(centers, Z, p=2.0)
        anc_idx = torch.argmin(d2_anc, dim=1)

        B_list: List[torch.Tensor] = []
        Bt_list: List[torch.Tensor] = []
        own_cuts: List[float] = []
        null_cuts: List[float] = []
        for v_idx, x in enumerate(X_list):
            x_sh = x - self.beta_u * (U_list[v_idx] @ Q_list[v_idx]) if self.use_variable_separation else x
            B_v, Bt_v = build_bipartite_affinity(x_sh, x_sh[anc_idx], k_anchor=k_a)
            _, _, _, own_tr, _ = bipartite_spectral_eigendecomposition(B_v, self.n_clusters)
            B_list.append(B_v)
            Bt_list.append(Bt_v)
            raw_cut = own_tr / max(1, self.n_clusters)
            if self.use_null_snr:
                c_n = _compute_anchor_null_cut(x_sh, k_anchor=k_a, n_anchors=m, n_clusters=self.n_clusters, seed=self.seed + 1000 + v_idx)
                null_cuts.append(c_n)
                own_cuts.append(raw_cut / c_n)
            else:
                null_cuts.append(1.0)
                own_cuts.append(raw_cut)

        w = torch.full((V,), 1.0 / V, dtype=torch.float32, device=self.device)
        I_err = torch.zeros((V,), dtype=torch.float32, device=self.device)
        W_loc = torch.full((m, V), 1.0 / V, dtype=torch.float32, device=self.device)
        I_loc = torch.zeros((m, V), dtype=torch.float32, device=self.device)
        B_S = sum(w[v] * B_list[v] for v in range(V))
        F, U_anc, Bt_S, spec_trace, gap = bipartite_spectral_eigendecomposition(B_S, self.n_clusters)
        mu = self.mu_init if self.use_spectral_control else 0.0

        weights_history: List[List[float]] = []
        lyapunov_history: List[float] = []
        spectral_gap_history: List[float] = []

        def compute_errors_and_lyap(
            cur_Z: torch.Tensor,
            cur_P: List[torch.Tensor],
            cur_U: List[Optional[torch.Tensor]],
            cur_Q: List[Optional[torch.Tensor]],
            cur_Bt: List[torch.Tensor],
            cur_own: List[float],
            cur_F: torch.Tensor,
            cur_w: torch.Tensor,
            cur_trace: float,
        ) -> Tuple[torch.Tensor, float]:
            errs = []
            lyap = 0.0
            c_float = float(self.n_clusters)
            for v_i, x in enumerate(X_list):
                recon = cur_Z @ cur_P[v_i]
                if cur_U[v_i] is not None and cur_Q[v_i] is not None:
                    recon = recon + cur_U[v_i] @ cur_Q[v_i]
                rec_rel = (float((x - recon).pow(2).sum().item()) / x_norms[v_i]) / dof_scales[v_i]
                # Exact Tr(F^T (I - Bt_v Bt_v^T) F) = c - ||Bt_v^T F||_F^2
                proj = cur_Bt[v_i].T @ cur_F
                cons_cut = max(0.0, (c_float - float(proj.pow(2).sum().item()))) / max(1, self.n_clusters)
                own_cut = cur_own[v_i]
                e_v = own_cut + self.gamma_cons * cons_cut + self.gamma_rec * rec_rel
                errs.append(e_v)
                lyap += float(cur_w[v_i].item()) * (rec_rel + self.alpha_graph * (own_cut + self.gamma_cons * cons_cut))
            entropy_term = float((cur_w * torch.log(cur_w.clamp_min(1e-12))).sum().item())
            spec_weight = self.mu_max if self.use_spectral_control else 0.0
            lyap += spec_weight * cur_trace + self.tau * entropy_term
            return torch.tensor(errs, dtype=torch.float32, device=self.device), lyap

        _, V_t = compute_errors_and_lyap(Z, P_list, U_list, Q_list, Bt_list, own_cuts, F, w, spec_trace)
        weights_history.append(w.detach().cpu().tolist())
        lyapunov_history.append(V_t)
        spectral_gap_history.append(gap)

        for t in range(self.max_iter):
            # Step A: Orthogonal Variable Separation update using O(n m r) bipartite operator Bt_S @ (Bt_S.T @ Z)
            grad_Z = self.lambda_z * (Bt_S @ (Bt_S.T @ Z))
            for v_i, x in enumerate(X_list):
                x_sh = x - (U_list[v_i] @ Q_list[v_i]) if self.use_variable_separation else x
                P_list[v_i] = Z.T @ x_sh
                grad_Z = grad_Z + (float(w[v_i].item()) / op_norm2[v_i]) * (x_sh @ P_list[v_i].T)

            Z_target = polar_orthogonalize(grad_Z)
            Z = polar_orthogonalize((1.0 - self.eta_z) * Z + self.eta_z * Z_target)

            for v_i, x in enumerate(X_list):
                if self.use_variable_separation:
                    res = x - Z @ (Z.T @ x)
                    r_v = U_list[v_i].shape[1]
                    U_cand, _ = fast_left_svd(res, r_v)
                    U_cand = U_cand - Z @ (Z.T @ U_cand)
                    U_cand = polar_orthogonalize(U_cand)
                    U_list[v_i] = U_cand
                    Q_list[v_i] = U_cand.T @ res
                    P_list[v_i] = Z.T @ (x - U_list[v_i] @ Q_list[v_i])
                else:
                    P_list[v_i] = Z.T @ x

            # Refresh bipartite graphs periodically
            if self.use_variable_separation and (t + 1) % 10 == 0 and t < self.max_iter - 5:
                for v_i, x in enumerate(X_list):
                    x_sh = x - self.beta_u * (U_list[v_i] @ Q_list[v_i])
                    B_new, _ = build_bipartite_affinity(x_sh, x_sh[anc_idx], k_anchor=k_a)
                    B_v = (1.0 - self.rho_a) * B_list[v_i] + self.rho_a * B_new
                    B_v = B_v / B_v.sum(dim=1, keepdim=True).clamp_min(1e-8)
                    col_deg = B_v.sum(dim=0).clamp_min(1e-8)
                    B_list[v_i] = B_v
                    Bt_list[v_i] = B_v * col_deg.rsqrt()[None, :]
                    _, _, _, own_tr, _ = bipartite_spectral_eigendecomposition(B_v, self.n_clusters)
                    own_cuts[v_i] = (own_tr / max(1, self.n_clusters)) / null_cuts[v_i]

            # Step B: Consensus bipartite affinity B_S + Dual spectral-gap feedback
            if self.use_local_pi:
                B_weighted = sum(B_list[v] * W_loc[:, v][None, :] for v in range(V))
            else:
                B_weighted = sum(w[v] * B_list[v] for v in range(V))
            if self.use_spectral_control and t >= self.warmup_iters and mu > 0.0:
                F_u = F / F.norm(dim=1, keepdim=True).clamp_min(1e-8)
                U_u = U_anc / U_anc.norm(dim=1, keepdim=True).clamp_min(1e-8)
                H_anc = torch.cdist(F_u, U_u, p=2.0).pow(2)
                B_tgt = (B_weighted - (0.14 * mu / k_a) * H_anc).clamp_min(0.0) + 1e-5 * B_weighted
            else:
                B_tgt = B_weighted.clamp_min(0.0)
            B_tgt = B_tgt / B_tgt.sum(dim=1, keepdim=True).clamp_min(1e-8)
            B_S = (1.0 - self.eta_s) * B_S + self.eta_s * B_tgt
            B_S = B_S / B_S.sum(dim=1, keepdim=True).clamp_min(1e-8)

            F, U_anc, Bt_S, spec_trace, gap = bipartite_spectral_eigendecomposition(B_S, self.n_clusters)
            if self.use_spectral_control and t >= self.warmup_iters:
                mu = float(np.clip(mu + self.eta_mu * (spec_trace - self.eps_spec), 0.0, self.mu_max))

            # Step C: Closed-loop global or Anchor-Local PI modality weight controller
            e_vec, _ = compute_errors_and_lyap(Z, P_list, U_list, Q_list, Bt_list, own_cuts, F, w, spec_trace)
            if self.use_feedback:
                if self.use_local_pi:
                    F_u_now = F / F.norm(dim=1, keepdim=True).clamp_min(1e-8)
                    F_anc = (B_S.T @ F_u_now) / B_S.sum(dim=0).clamp_min(1e-8)[:, None]
                    F_anc_u = F_anc / F_anc.norm(dim=1, keepdim=True).clamp_min(1e-8)
                    H_now = torch.cdist(F_u_now, F_anc_u, p=2.0).pow(2)
                    loc_cols = []
                    for v_i in range(V):
                        c_sum = B_list[v_i].sum(dim=0)
                        raw_l = 0.5 * (B_list[v_i] * H_now).sum(dim=0) / c_sum.clamp_min(1e-6)
                        fb_val = float(raw_l[c_sum > 1e-4].mean().item()) if (c_sum > 1e-4).any() else 0.0
                        loc_cols.append(torch.where(c_sum > 1e-4, raw_l, torch.full_like(raw_l, fb_val)))
                    loc_mat = torch.stack(loc_cols, dim=1)  # (m, V)
                    # Double-center across anchors (zero global bias) and views, then bound deviations
                    loc_c = loc_mat - loc_mat.mean(dim=0, keepdim=True)
                    loc_dev = torch.clamp(loc_c - loc_c.mean(dim=1, keepdim=True), -0.25, 0.25)
                    e_loc = e_vec[None, :] + self.alpha_local * loc_dev
                    W_loc, I_loc = _apply_deadzone_pi(
                        e_loc, I_loc, W_loc,
                        dead_zone=self.dead_zone,
                        beta_i=self.beta_i,
                        kappa_i=self.kappa_i,
                        eta_w=self.eta_w,
                        tau=self.tau,
                    )
                    w = W_loc.mean(dim=0)
                else:
                    w, I_err = _apply_deadzone_pi(
                        e_vec, I_err, w,
                        dead_zone=self.dead_zone,
                        beta_i=self.beta_i,
                        kappa_i=self.kappa_i,
                        eta_w=self.eta_w,
                        tau=self.tau,
                    )

            _, V_t = compute_errors_and_lyap(Z, P_list, U_list, Q_list, Bt_list, own_cuts, F, w, spec_trace)
            weights_history.append(w.detach().cpu().tolist())
            lyapunov_history.append(V_t)
            spectral_gap_history.append(gap)

        F_np = F.detach().cpu().numpy()
        row_norms = np.linalg.norm(F_np, axis=1, keepdims=True)
        row_norms = np.where(row_norms < 1e-12, 1.0, row_norms)
        F_unit = F_np / row_norms
        labels = KMeans(n_clusters=self.n_clusters, n_init=20 if n <= 15000 else 10, random_state=self.seed).fit_predict(F_unit)

        res_dict = {
            "labels": labels,
            "Z": Z.detach().cpu().numpy(),
            "F": F_unit,
            "final_weights": w.detach().cpu().numpy(),
            "weights_history": np.array(weights_history),
            "lyapunov_history": np.array(lyapunov_history),
            "spectral_gap_history": np.array(spectral_gap_history),
        }
        if self.use_local_pi:
            res_dict["local_weights"] = W_loc.detach().cpu().numpy()
        return res_dict


"""Baselines, deep multi-omics models, partial/scalable benchmarks, and evaluation metrics."""

from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F_nn
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.mixture import GaussianMixture
from lifelines.statistics import multivariate_logrank_test

# (uses CL-VSSC helpers defined above)


def _cluster_embeddings(F_np: np.ndarray, n_clusters: int, seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    norms = np.linalg.norm(F_np, axis=1, keepdims=True)
    norms = np.where(norms < 1e-12, 1.0, norms)
    F_unit = F_np / norms
    n, d = F_unit.shape
    if n <= 5000:
        labels = KMeans(n_clusters=n_clusters, n_init=30, random_state=seed).fit_predict(F_unit)
    elif n <= 60000 and d <= 64:
        labels = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit_predict(F_unit)
    else:
        labels = MiniBatchKMeans(
            n_clusters=n_clusters, n_init=10, batch_size=min(4096, n), random_state=seed
        ).fit_predict(F_unit)
    return labels, F_unit


def _njw_cluster(S: torch.Tensor, n_clusters: int, seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    F, _, _, _ = spectral_eigendecomposition(S, n_clusters)
    return _cluster_embeddings(F.detach().cpu().numpy(), n_clusters, seed=seed)


def clustering_acc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Unsupervised clustering accuracy (ACC) via optimal Hungarian assignment."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    _, t_inv = np.unique(y_true, return_inverse=True)
    _, p_inv = np.unique(y_pred, return_inverse=True)
    dim = max(int(t_inv.max()) + 1, int(p_inv.max()) + 1)
    w = np.zeros((dim, dim), dtype=np.int64)
    np.add.at(w, (p_inv, t_inv), 1)
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    return float(w[row_ind, col_ind].sum() / max(1, len(y_true)))


@torch.no_grad()
def run_concat_spectral(
    views: List[np.ndarray], n_clusters: int, seed: int = 42, device: Optional[str] = None
) -> Dict[str, object]:
    dev = get_device(device)
    X_list, _ = preprocess_views(views, dev)
    X_cat = torch.cat(X_list, dim=1)
    n = X_cat.shape[0]
    k_nn = max(8, min(25, n // 8))
    S = build_knn_affinity(X_cat, k=k_nn)
    labels, emb = _njw_cluster(S, n_clusters, seed=seed)
    return {"labels": labels, "S": S.detach().cpu().numpy(), "F": emb}


@torch.no_grad()
def run_snf(
    views: List[np.ndarray],
    n_clusters: int,
    k: Optional[int] = None,
    t_iter: int = 20,
    seed: int = 42,
    device: Optional[str] = None,
) -> Dict[str, object]:
    """Similarity Network Fusion (Wang et al., Nature Methods 2014)."""
    dev = get_device(device)
    X_list, _ = preprocess_views(views, dev)
    V = len(X_list)
    n = X_list[0].shape[0]
    k_eff = k if k is not None else max(8, min(20, n // 10))

    P_list, S_list = [], []
    for x in X_list:
        dist = torch.cdist(x, x, p=2.0)
        d_sorted, idx = torch.sort(dist, dim=1)
        eps_i = d_sorted[:, 1 : k_eff + 1].mean(dim=1).clamp_min(1e-6)
        sigma = 0.5 * (eps_i[:, None] + eps_i[None, :] + dist) / 3.0 + 1e-6
        W = torch.exp(-(dist.pow(2)) / (0.5 * sigma.pow(2) + 1e-6))
        W = 0.5 * (W + W.T)

        W_off = W.clone()
        W_off.fill_diagonal_(0.0)
        row_s = W_off.sum(dim=1, keepdim=True).clamp_min(1e-8)
        P = 0.5 * (W_off / row_s)
        P.fill_diagonal_(0.5)
        P = 0.5 * (P + P.T)
        P_list.append(P)

        mask = torch.zeros((n, n), dtype=torch.bool, device=dev)
        row_ar = torch.arange(n, device=dev)[:, None]
        mask[row_ar, idx[:, 1 : k_eff + 1]] = True
        W_loc = W_off * mask.float()
        loc_s = W_loc.sum(dim=1, keepdim=True).clamp_min(1e-8)
        S_loc = W_loc / loc_s
        S_list.append(S_loc)

    if V > 1:
        for _ in range(t_iter):
            P_next = []
            for v in range(V):
                other_mean = sum(P_list[u] for u in range(V) if u != v) / (V - 1)
                P_new = S_list[v] @ other_mean @ S_list[v].T
                P_new = 0.5 * (P_new + P_new.T)
                row_s = P_new.sum(dim=1, keepdim=True).clamp_min(1e-8)
                P_new = P_new / row_s
                P_new.fill_diagonal_(0.5)
                P_new = 0.5 * (P_new + P_new.T)
                P_next.append(P_new)
            P_list = P_next

    W_fused = sum(P_list) / V
    W_fused.fill_diagonal_(0.0)
    W_fused = doubly_stochastic_normalize(W_fused)
    labels, emb = _njw_cluster(W_fused, n_clusters, seed=seed)
    return {"labels": labels, "S": W_fused.detach().cpu().numpy(), "F": emb}


@torch.no_grad()
def run_nemo(
    views: List[np.ndarray],
    n_clusters: int,
    k: Optional[int] = None,
    masks: Optional[List[np.ndarray]] = None,
    seed: int = 42,
    device: Optional[str] = None,
) -> Dict[str, object]:
    """Neighborhood-based Multi-Omics clustering with exact partial-view support (Rappoport & Shamir, 2019)."""
    dev = get_device(device)
    has_missing = masks is not None
    X_list, M_list = preprocess_views(views, dev, masks=masks)
    V = len(X_list)
    n = X_list[0].shape[0]
    k_eff = k if k is not None else max(8, min(25, n // 6))

    RS_list = []
    pair_masks = []
    for v_i, x in enumerate(X_list):
        if has_missing:
            obs_idx = torch.nonzero(M_list[v_i] > 0.5, as_tuple=False).squeeze(1)
            rs_full = torch.zeros((n, n), dtype=torch.float32, device=dev)
            if len(obs_idx) >= 3:
                x_sub = x[obs_idx]
                k_sub = min(k_eff, len(obs_idx) - 1)
                dist2 = torch.cdist(x_sub, x_sub, p=2.0).pow(2)
                dist2.fill_diagonal_(float("inf"))
                knn_vals, knn_idx = torch.topk(dist2, k=k_sub, dim=1, largest=False)
                sig_i = knn_vals.mean(dim=1).clamp_min(1e-6)
                dist2.fill_diagonal_(0.0)
                denom = sig_i[:, None] + sig_i[None, :] + 1e-6
                W = torch.exp(-dist2 / denom)
                mask = torch.zeros((len(obs_idx), len(obs_idx)), dtype=torch.bool, device=dev)
                row_ar = torch.arange(len(obs_idx), device=dev)[:, None]
                mask[row_ar, knn_idx] = True
                mask = mask | mask.T
                mask.fill_diagonal_(False)
                W_knn = W * mask.float()
                rs_sub = W_knn / W_knn.sum(dim=1, keepdim=True).clamp_min(1e-8)
                rs_sub = 0.5 * (rs_sub + rs_sub.T)
                rs_full[obs_idx[:, None], obs_idx[None, :]] = rs_sub
            RS_list.append(rs_full)
            pair_masks.append(M_list[v_i][:, None] * M_list[v_i][None, :])
        else:
            dist2 = torch.cdist(x, x, p=2.0).pow(2)
            dist2.fill_diagonal_(float("inf"))
            knn_vals, knn_idx = torch.topk(dist2, k=min(k_eff, n - 1), dim=1, largest=False)
            sig_i = knn_vals.mean(dim=1).clamp_min(1e-6)
            dist2.fill_diagonal_(0.0)
            denom = sig_i[:, None] + sig_i[None, :] + 1e-6
            W = torch.exp(-dist2 / denom)

            mask = torch.zeros((n, n), dtype=torch.bool, device=dev)
            row_ar = torch.arange(n, device=dev)[:, None]
            mask[row_ar, knn_idx] = True
            mask = mask | mask.T
            mask.fill_diagonal_(False)
            W_knn = W * mask.float()
            rs = W_knn / W_knn.sum(dim=1, keepdim=True).clamp_min(1e-8)
            RS_list.append(0.5 * (rs + rs.T))

    if has_missing:
        num = sum(RS_list)
        den = sum(pair_masks)
        ARS = torch.where(den > 0.5, num / den.clamp_min(1.0), torch.zeros_like(num))
        ARS.fill_diagonal_(0.0)
        empty_rows = ARS.sum(dim=1) < 1e-8
        if empty_rows.any():
            ARS[empty_rows] = 1.0 / max(1, n - 1)
            ARS.fill_diagonal_(0.0)
    else:
        ARS = sum(RS_list) / V

    ARS = doubly_stochastic_normalize(ARS)
    labels, emb = _njw_cluster(ARS, n_clusters, seed=seed)
    return {"labels": labels, "S": ARS.detach().cpu().numpy(), "F": emb}


@torch.no_grad()
def run_openloop_mavs(
    views: List[np.ndarray],
    n_clusters: int,
    shared_dim: int = 24,
    spec_dim: int = 8,
    n_iters: int = 20,
    seed: int = 42,
    device: Optional[str] = None,
) -> Dict[str, object]:
    """Open-loop variable separation + static/attention fusion + post-hoc spectral clustering."""
    dev = get_device(device)
    torch.manual_seed(seed)
    X_list, _ = preprocess_views(views, dev)
    V = len(X_list)
    n = X_list[0].shape[0]
    k_nn = max(8, min(25, n // 8))

    X_cat = torch.cat(X_list, dim=1)
    r = min(shared_dim, n - 1, X_cat.shape[1])
    Z, _ = fast_left_svd(X_cat, r)

    U_list, Q_list, P_list = [], [], []
    for x in X_list:
        r_v = max(1, min(spec_dim, n - r - 1 if n - r > 2 else max(1, n // 6), x.shape[1]))
        p_v = Z.T @ x
        res = x - Z @ p_v
        res = res - Z @ (Z.T @ res)
        u_v, _ = fast_left_svd(res, r_v)
        U_list.append(u_v)
        Q_list.append(u_v.T @ res)
        P_list.append(p_v)

    for _ in range(n_iters):
        grad_Z = torch.zeros_like(Z)
        for v_i, x in enumerate(X_list):
            x_sh = x - U_list[v_i] @ Q_list[v_i]
            P_list[v_i] = Z.T @ x_sh
            grad_Z = grad_Z + x_sh @ P_list[v_i].T
        u_z, _, vt_z = torch.linalg.svd(grad_Z, full_matrices=False)
        Z = u_z @ vt_z
        for v_i, x in enumerate(X_list):
            res = x - Z @ (Z.T @ x)
            r_v = U_list[v_i].shape[1]
            u_v, _ = fast_left_svd(res, r_v)
            u_v = u_v - Z @ (Z.T @ u_v)
            u_p, _, vt_p = torch.linalg.svd(u_v, full_matrices=False)
            U_list[v_i] = u_p @ vt_p
            Q_list[v_i] = U_list[v_i].T @ res
            P_list[v_i] = Z.T @ (x - U_list[v_i] @ Q_list[v_i])

    energies = []
    A_list = []
    for v_i, x in enumerate(X_list):
        x_sh = x - U_list[v_i] @ Q_list[v_i]
        A_list.append(build_knn_affinity(x_sh, k=k_nn))
        en = float((Z @ P_list[v_i]).pow(2).sum().item()) / (float(x.pow(2).sum().item()) + 1e-8)
        energies.append(en)
    w_open = torch.softmax(torch.tensor(energies, dtype=torch.float32, device=dev) / 0.25, dim=0)

    S_fused = sum(w_open[v] * A_list[v] for v in range(V))
    S_fused = doubly_stochastic_normalize(S_fused)
    labels, emb = _njw_cluster(S_fused, n_clusters, seed=seed)
    return {
        "labels": labels,
        "S": S_fused.detach().cpu().numpy(),
        "F": emb,
        "final_weights": w_open.detach().cpu().numpy(),
    }


@torch.no_grad()
def run_mcgc(
    views: List[np.ndarray],
    n_clusters: int,
    k: Optional[int] = None,
    n_iters: int = 25,
    seed: int = 42,
    device: Optional[str] = None,
) -> Dict[str, object]:
    """Multiview Consensus Graph Clustering with inverse-disagreement weights and CLR rank penalty."""
    dev = get_device(device)
    torch.manual_seed(seed)
    X_list, _ = preprocess_views(views, dev)
    V = len(X_list)
    n = X_list[0].shape[0]
    k_eff = k if k is not None else max(8, min(25, n // 6))

    A_list = [build_knn_affinity(x, k=k_eff) for x in X_list]
    w = torch.full((V,), 1.0 / V, dtype=torch.float32, device=dev)
    S = sum(w[v] * A_list[v] for v in range(V))
    S = 0.5 * (S + S.T)
    mu = 0.05

    for _ in range(n_iters):
        F, _, spec_trace, _ = spectral_eigendecomposition(S, n_clusters)
        f_norms = F.norm(dim=1, keepdim=True).clamp_min(1e-8)
        H = torch.cdist(F / f_norms, F / f_norms, p=2.0).pow(2)

        diffs = torch.stack([torch.norm(S - A_v, p="fro").clamp_min(1e-6) for A_v in A_list])
        inv_d = 1.0 / diffs
        w = inv_d / inv_d.sum().clamp_min(1e-8)

        S_weighted = sum(w[v] * A_list[v] for v in range(V))
        S_target = (S_weighted - (mu / k_eff) * H).clamp_min(0.0)
        S_target.fill_diagonal_(0.0)
        row_st = S_target.sum(dim=1, keepdim=True).clamp_min(1e-8)
        S_target = S_target / row_st
        S_target = 0.5 * (S_target + S_target.T)
        S = 0.65 * S + 0.35 * S_target
        S = 0.5 * (S + S.T)

        if spec_trace > 0.02:
            mu = min(mu * 1.08, 0.20)

    S = doubly_stochastic_normalize(S)
    labels, emb = _njw_cluster(S, n_clusters, seed=seed)
    return {
        "labels": labels,
        "S": S.detach().cpu().numpy(),
        "F": emb,
        "final_weights": w.detach().cpu().numpy(),
    }


def run_subtype_gan(
    views: List[np.ndarray],
    n_clusters: int,
    latent_dim: int = 24,
    epochs: Optional[int] = None,
    lr: float = 1e-3,
    seed: int = 42,
    device: Optional[str] = None,
) -> Dict[str, object]:
    """Adversarial multi-omics autoencoder baseline inspired by Subtype-GAN (Yang et al., 2021)."""
    dev = get_device(device)
    torch.manual_seed(seed)
    np.random.seed(seed)
    X_list, _ = preprocess_views(views, dev)
    n = X_list[0].shape[0]
    dims = [x.shape[1] for x in X_list]
    n_epochs = epochs if epochs is not None else (100 if n <= 2000 else 50)
    batch_size = min(1024, n)

    class _SubtypeGANNet(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.encs = nn.ModuleList(
                [nn.Sequential(nn.Linear(d, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Linear(128, 64)) for d in dims]
            )
            self.to_z = nn.Linear(64, latent_dim)
            self.decs = nn.ModuleList(
                [nn.Sequential(nn.Linear(latent_dim, 128), nn.LeakyReLU(0.2), nn.Linear(128, d)) for d in dims]
            )

        def encode(self, xs: List[torch.Tensor]) -> torch.Tensor:
            hs = torch.stack([enc(x) for enc, x in zip(self.encs, xs)], dim=0).mean(dim=0)
            return self.to_z(F_nn.leaky_relu(hs, 0.2))

        def decode(self, z: torch.Tensor) -> List[torch.Tensor]:
            return [dec(z) for dec in self.decs]

    class _Disc(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(latent_dim, 64), nn.LeakyReLU(0.2), nn.Linear(64, 32), nn.LeakyReLU(0.2), nn.Linear(32, 1)
            )

        def forward(self, z: torch.Tensor) -> torch.Tensor:
            return self.net(z)

    gen = _SubtypeGANNet().to(dev)
    disc = _Disc().to(dev)
    opt_g = torch.optim.Adam(gen.parameters(), lr=lr, weight_decay=1e-5)
    opt_d = torch.optim.Adam(disc.parameters(), lr=lr, weight_decay=1e-5)
    bce = nn.BCEWithLogitsLoss()

    with torch.enable_grad():
        gen.train()
        disc.train()
        for _ in range(n_epochs):
            perm = torch.randperm(n, device=dev)
            for start in range(0, n, batch_size):
                idx = perm[start : start + batch_size]
                if len(idx) < 4:
                    continue
                xb = [x[idx] for x in X_list]
                z_fake = gen.encode(xb).detach()
                z_real = torch.randn_like(z_fake)
                opt_d.zero_grad()
                loss_d = 0.5 * (
                    bce(disc(z_real), torch.ones((len(idx), 1), device=dev))
                    + bce(disc(z_fake), torch.zeros((len(idx), 1), device=dev))
                )
                loss_d.backward()
                opt_d.step()

                opt_g.zero_grad()
                z = gen.encode(xb)
                recons = gen.decode(z)
                rec_loss = sum(F_nn.mse_loss(rh, x_v) for rh, x_v in zip(recons, xb))
                adv_loss = bce(disc(z), torch.ones((len(idx), 1), device=dev))
                loss_g = rec_loss + 0.05 * adv_loss
                loss_g.backward()
                opt_g.step()

    with torch.no_grad():
        gen.eval()
        zs = []
        for start in range(0, n, 2048):
            zs.append(gen.encode([x[start : start + 2048] for x in X_list]).cpu())
        Z_np = torch.cat(zs, dim=0).numpy()

    if n <= 5000:
        try:
            labels = GaussianMixture(n_components=n_clusters, covariance_type="diag", n_init=5, random_state=seed).fit_predict(Z_np)
        except Exception:
            labels, _ = _cluster_embeddings(Z_np, n_clusters, seed=seed)
        norms = np.linalg.norm(Z_np, axis=1, keepdims=True)
        F_unit = Z_np / np.where(norms < 1e-12, 1.0, norms)
    else:
        labels, F_unit = _cluster_embeddings(Z_np, n_clusters, seed=seed)
    return {"labels": labels, "F": F_unit}


def run_attention_moae(
    views: List[np.ndarray],
    n_clusters: int,
    latent_dim: int = 24,
    epochs: Optional[int] = None,
    lr: float = 1e-3,
    seed: int = 42,
    device: Optional[str] = None,
) -> Dict[str, object]:
    """Attention-gated multi-omics autoencoder baseline inspired by MOMA (Moon & Lee, 2022)."""
    dev = get_device(device)
    torch.manual_seed(seed)
    np.random.seed(seed)
    X_list, _ = preprocess_views(views, dev)
    n = X_list[0].shape[0]
    dims = [x.shape[1] for x in X_list]
    n_epochs = epochs if epochs is not None else (100 if n <= 2000 else 50)
    batch_size = min(1024, n)

    class _AttMOAE(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.encs = nn.ModuleList(
                [nn.Sequential(nn.Linear(d, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Linear(128, latent_dim)) for d in dims]
            )
            self.att_w = nn.Linear(latent_dim, 32)
            self.att_v = nn.Linear(32, 1, bias=False)
            self.decs = nn.ModuleList(
                [nn.Sequential(nn.Linear(latent_dim, 128), nn.GELU(), nn.Linear(128, d)) for d in dims]
            )

        def forward(self, xs: List[torch.Tensor]) -> Tuple[torch.Tensor, List[torch.Tensor], torch.Tensor]:
            hs = torch.stack([enc(x) for enc, x in zip(self.encs, xs)], dim=1)  # (b, V, d)
            scores = self.att_v(torch.tanh(self.att_w(hs))).squeeze(-1)  # (b, V)
            alpha = torch.softmax(scores, dim=1)
            z = torch.sum(hs * alpha.unsqueeze(-1), dim=1)
            recons = [dec(z) for dec in self.decs]
            return z, recons, alpha

    model = _AttMOAE().to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    with torch.enable_grad():
        model.train()
        for _ in range(n_epochs):
            perm = torch.randperm(n, device=dev)
            for start in range(0, n, batch_size):
                idx = perm[start : start + batch_size]
                if len(idx) < 4:
                    continue
                xb = [x[idx] for x in X_list]
                opt.zero_grad()
                _, recons, _ = model(xb)
                loss = sum(F_nn.mse_loss(rh, x_v) for rh, x_v in zip(recons, xb))
                loss.backward()
                opt.step()

    with torch.no_grad():
        model.eval()
        zs, alphas = [], []
        for start in range(0, n, 2048):
            z_b, _, al_b = model([x[start : start + 2048] for x in X_list])
            zs.append(z_b.cpu())
            alphas.append(al_b.cpu())
        Z_np = torch.cat(zs, dim=0).numpy()
        mean_w = torch.cat(alphas, dim=0).mean(dim=0).numpy()

    labels, F_unit = _cluster_embeddings(Z_np, n_clusters, seed=seed)
    return {"labels": labels, "F": F_unit, "final_weights": mean_w}


@torch.no_grad()
def run_wnn(
    views: List[np.ndarray],
    n_clusters: int,
    k: Optional[int] = None,
    n_anchors: int = 384,
    seed: int = 42,
    device: Optional[str] = None,
) -> Dict[str, object]:
    """Weighted Nearest Neighbors multimodal clustering (Hao et al., Cell 2021)."""
    dev = get_device(device)
    X_list, _ = preprocess_views(views, dev)
    V = len(X_list)
    n = X_list[0].shape[0]

    if n > 6000:
        # Scalable anchor-WNN via cross-view local landmark predictability ratios
        X_cat = torch.cat(X_list, dim=1)
        Z0, _ = fast_left_svd(X_cat, min(24, n - 1, X_cat.shape[1]))
        m = min(n_anchors, n // 4)
        km = MiniBatchKMeans(n_clusters=m, n_init=3, batch_size=min(2048, n), max_iter=25, random_state=seed)
        km.fit(Z0.cpu().numpy())
        centers = torch.tensor(km.cluster_centers_, dtype=torch.float32, device=dev)
        anc_idx = torch.argmin(torch.cdist(centers, Z0, p=2.0), dim=1)
        k_a = min(15, m)

        B_list = []
        ratio_scores = []
        for v_i, x in enumerate(X_list):
            B_v, _ = build_bipartite_affinity(x, x[anc_idx], k_anchor=k_a)
            B_list.append(B_v)
            pred_within = B_v @ x[anc_idx]
            err_within = (x - pred_within).pow(2).sum(dim=1).clamp_min(1e-6)
            cross_errs = []
            for u_i in range(V):
                if u_i == v_i:
                    continue
                pred_cross = B_list[u_i] @ x[anc_idx] if u_i < v_i else build_bipartite_affinity(X_list[u_i], X_list[u_i][anc_idx], k_anchor=k_a)[0] @ x[anc_idx]
                cross_errs.append((x - pred_cross).pow(2).sum(dim=1).clamp_min(1e-6))
            mean_cross = sum(cross_errs) / max(1, len(cross_errs))
            ratio_scores.append(torch.log(mean_cross / err_within).clamp(-5.0, 5.0))
        theta = torch.softmax(torch.stack(ratio_scores, dim=1), dim=1)  # (n, V)
        B_wnn = sum(theta[:, v : v + 1] * B_list[v] for v in range(V))
        F, _, _, _, _ = bipartite_spectral_eigendecomposition(B_wnn, n_clusters)
        labels, emb = _cluster_embeddings(F.detach().cpu().numpy(), n_clusters, seed=seed)
        return {"labels": labels, "F": emb, "final_weights": theta.mean(dim=0).detach().cpu().numpy()}

    k_eff = k if k is not None else max(10, min(25, n // 8))
    K_list = [build_knn_affinity(x, k=k_eff) for x in X_list]
    aff_scores = []
    for v_i, x in enumerate(X_list):
        pred_within = K_list[v_i] @ x
        err_within = (x - pred_within).pow(2).sum(dim=1).clamp_min(1e-6)
        cross_errs = []
        for u_i in range(V):
            if u_i != v_i:
                cross_errs.append((x - K_list[u_i] @ x).pow(2).sum(dim=1).clamp_min(1e-6))
        mean_cross = sum(cross_errs) / max(1, len(cross_errs))
        aff_scores.append(torch.log(mean_cross / err_within).clamp(-5.0, 5.0))
    theta = torch.softmax(torch.stack(aff_scores, dim=1), dim=1)  # (n, V)
    W_wnn = sum(theta[:, v : v + 1] * K_list[v] for v in range(V))
    W_wnn.fill_diagonal_(0.0)
    W_wnn = doubly_stochastic_normalize(0.5 * (W_wnn + W_wnn.T))
    labels, emb = _njw_cluster(W_wnn, n_clusters, seed=seed)
    return {
        "labels": labels,
        "S": W_wnn.detach().cpu().numpy(),
        "F": emb,
        "final_weights": theta.mean(dim=0).detach().cpu().numpy(),
    }


@torch.no_grad()
def run_lmvsc(
    views: List[np.ndarray],
    n_clusters: int,
    n_anchors: int = 384,
    anchor_k: int = 15,
    seed: int = 42,
    device: Optional[str] = None,
) -> Dict[str, object]:
    """Large-scale Multi-View Subspace Clustering in linear time (Kang et al., AAAI 2020)."""
    dev = get_device(device)
    torch.manual_seed(seed)
    X_list, _ = preprocess_views(views, dev)
    V = len(X_list)
    n = X_list[0].shape[0]
    m = min(n_anchors, max(n_clusters + 4, n // 4))
    k_a = min(anchor_k, m)

    Bt_views = []
    for v_i, x in enumerate(X_list):
        # View-specific or shared landmark selection via fast KMeans on reduced projection
        r_tmp = min(20, x.shape[1], n - 1)
        U_tmp, _ = fast_left_svd(x, r_tmp)
        km = MiniBatchKMeans(n_clusters=m, n_init=3, batch_size=min(2048, n), max_iter=25, random_state=seed + v_i)
        km.fit(U_tmp.cpu().numpy())
        centers = torch.tensor(km.cluster_centers_, dtype=torch.float32, device=dev)
        anc_idx = torch.argmin(torch.cdist(centers, U_tmp, p=2.0), dim=1)
        _, Bt_v = build_bipartite_affinity(x, x[anc_idx], k_anchor=k_a)
        Bt_views.append(Bt_v)

    H_cat = torch.cat(Bt_views, dim=1) / np.sqrt(float(V))
    F, _ = fast_left_svd(H_cat, n_clusters)
    labels, emb = _cluster_embeddings(F.detach().cpu().numpy(), n_clusters, seed=seed)
    return {"labels": labels, "F": emb}


def corrupt_view(
    views: List[np.ndarray],
    view_idx: int,
    noise_std: float = 0.0,
    dropout_rate: float = 0.0,
    seed: int = 0,
) -> List[np.ndarray]:
    """Inject scale-invariant Gaussian noise and feature dropout into one modality."""
    rng = np.random.default_rng(seed)
    out = [np.array(v, copy=True, dtype=np.float32) for v in views]
    target = out[view_idx]
    feat_std = target.std(axis=0, keepdims=True)
    feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)
    if noise_std > 0.0:
        target = target + rng.normal(0.0, noise_std, size=target.shape).astype(np.float32) * feat_std
    if dropout_rate > 0.0:
        mask = rng.random(size=target.shape) >= dropout_rate
        target = target * mask.astype(np.float32)
    out[view_idx] = target
    return out


def _logrank_stat_batch(
    obs_minus_exp_vec: np.ndarray,
    diag_w: np.ndarray,
    R_scaled: np.ndarray,
    G_batch: np.ndarray,
) -> np.ndarray:
    """Compute exact multivariate log-rank chi-square statistics for a batch of one-hot label matrices."""
    B, n, K = G_batch.shape
    Z = np.einsum("n,bnk->bk", obs_minus_exp_vec, G_batch)
    diag_val = np.einsum("n,bnk->bk", diag_w, G_batch)
    U = np.einsum("mn,bnk->bmk", R_scaled, G_batch)
    off = np.einsum("bmk,bml->bkl", U, U)
    V = -off
    idx = np.arange(K)
    V[:, idx, idx] += diag_val

    Z_red = Z[:, :-1]
    V_red = V[:, :-1, :-1]
    stats = np.zeros(B, dtype=np.float64)
    try:
        sol = np.linalg.solve(V_red, Z_red[..., None])[..., 0]
        stats = np.sum(Z_red * sol, axis=1)
    except np.linalg.LinAlgError:
        for b in range(B):
            try:
                stats[b] = float(Z_red[b] @ np.linalg.pinv(V_red[b]) @ Z_red[b])
            except Exception:
                stats[b] = 0.0
    return np.nan_to_num(stats, nan=0.0, posinf=0.0, neginf=0.0)


def permutation_logrank_p(
    times: np.ndarray,
    events: np.ndarray,
    labels: np.ndarray,
    n_perm: int = 5000,
    seed: int = 42,
) -> Tuple[float, float]:
    """Empirical exact permutation log-rank test (Rappoport & Shamir, NAR 2018)."""
    times = np.asarray(times, dtype=np.float64)
    events = np.asarray(events, dtype=np.float64)
    labels = np.asarray(labels)
    valid = np.isfinite(times) & np.isfinite(events) & (times > 0)
    times, events, labels = times[valid], events[valid], labels[valid]
    _, inv = np.unique(labels, return_inverse=True)
    K = int(inv.max()) + 1 if len(inv) > 0 else 0
    t_ev = np.unique(times[events > 0.5])
    if K < 2 or len(t_ev) == 0:
        return 0.0, 1.0

    R = (times[None, :] >= t_ev[:, None]).astype(np.float64)
    N_m = np.maximum(R.sum(axis=1), 1e-12)
    O_m = np.array([events[times == t].sum() for t in t_ev], dtype=np.float64)
    factor_m = np.where(N_m > 1.0, O_m * (N_m - O_m) / np.maximum(N_m - 1.0, 1.0), 0.0)

    obs_minus_exp_vec = events - ((O_m / N_m) @ R)
    diag_w = (factor_m / N_m) @ R
    R_scaled = (np.sqrt(np.maximum(factor_m, 0.0)) / N_m)[:, None] * R

    n = len(times)
    G_obs = np.zeros((1, n, K), dtype=np.float64)
    G_obs[0, np.arange(n), inv] = 1.0
    T_obs = float(_logrank_stat_batch(obs_minus_exp_vec, diag_w, R_scaled, G_obs)[0])

    rng = np.random.default_rng(seed)
    G_perm_base = G_obs[0]
    ge_count = 0
    batch_size = 500
    done = 0
    while done < n_perm:
        bs = min(batch_size, n_perm - done)
        perms = np.argsort(rng.random((bs, n)), axis=1)
        G_batch = G_perm_base[perms]
        T_perm = _logrank_stat_batch(obs_minus_exp_vec, diag_w, R_scaled, G_batch)
        ge_count += int(np.sum(T_perm >= T_obs - 1e-12))
        done += bs

    perm_p = float((1 + ge_count) / (1 + n_perm))
    return T_obs, perm_p


def evaluate_clustering(
    labels: np.ndarray,
    true_labels: Optional[np.ndarray] = None,
    survival_df: Optional[pd.DataFrame] = None,
    embedding: Optional[np.ndarray] = None,
    compute_perm: bool = False,
    n_perm: int = 5000,
    perm_seed: int = 42,
) -> Dict[str, float]:
    """Compute survival log-rank, permutation p-value, silhouette, ARI/NMI/ACC, and cluster balance."""
    labels = np.asarray(labels)
    _, counts = np.unique(labels, return_counts=True)
    res: Dict[str, float] = {
        "n_clusters_found": float(len(counts)),
        "min_cluster_size": float(counts.min()),
        "max_cluster_size": float(counts.max()),
    }
    if embedding is not None and len(counts) > 1 and len(labels) > len(counts):
        s_size = min(2500, len(labels)) if len(labels) > 3000 else None
        res["silhouette"] = float(silhouette_score(embedding, labels, sample_size=s_size, random_state=42))
    if true_labels is not None:
        res["ari"] = float(adjusted_rand_score(true_labels, labels))
        res["nmi"] = float(normalized_mutual_info_score(true_labels, labels))
        res["acc"] = clustering_acc(true_labels, labels)
    if survival_df is not None and len(counts) > 1:
        df = survival_df.copy()
        df["cluster"] = labels
        df = df.dropna(subset=["time", "event"])
        df = df[df["time"] > 0]
        if df["cluster"].nunique() > 1:
            lr = multivariate_logrank_test(df["time"], df["cluster"], df["event"])
            p = max(float(lr.p_value), 1e-300)
            res["logrank_p"] = float(lr.p_value)
            res["neg_log10_p"] = float(-np.log10(p))
            if compute_perm:
                _, perm_p = permutation_logrank_p(
                    df["time"].to_numpy(),
                    df["event"].to_numpy(),
                    df["cluster"].to_numpy(),
                    n_perm=n_perm,
                    seed=perm_seed,
                )
                res["perm_p"] = perm_p
                res["neg_log10_perm_p"] = float(-np.log10(max(perm_p, 1e-12)))
        else:
            res["logrank_p"] = 1.0
            res["neg_log10_p"] = 0.0
            if compute_perm:
                res["perm_p"] = 1.0
                res["neg_log10_perm_p"] = 0.0
    elif survival_df is not None:
        res["logrank_p"] = 1.0
        res["neg_log10_p"] = 0.0
        if compute_perm:
            res["perm_p"] = 1.0
            res["neg_log10_perm_p"] = 0.0
    return res


"""Verified downloader and preprocessor for the Rappoport & Shamir 10-TCGA multi-omics benchmark."""

from pathlib import Path
from typing import Dict, List, Tuple
import re
import shutil
import subprocess
import zipfile

import numpy as np
import pandas as pd


BASE_URL = "https://acgt.cs.tau.ac.il/multi_omic_benchmark/data"

COHORTS = ["aml", "bic", "coad", "gbm", "kirc", "lihc", "lusc", "ov", "sarc", "skcm"]

COHORT_ARCHIVES: Dict[str, str] = {
    "aml": "aml.zip",
    "bic": "breast.zip",
    "coad": "colon.zip",
    "gbm": "gbm.zip",
    "kirc": "kidney.zip",
    "lihc": "liver.zip",
    "lusc": "lung.zip",
    "ov": "ovarian.zip",
    "sarc": "sarcoma.zip",
    "skcm": "melanoma.zip",
}

CANONICAL_K: Dict[str, int] = {
    "aml": 4,
    "bic": 5,
    "coad": 4,
    "gbm": 4,
    "kirc": 4,
    "lihc": 4,
    "lusc": 4,
    "ov": 4,
    "sarc": 4,
    "skcm": 4,
}

_SAMPLE_TYPE_PRIORITY = {"01": 0, "03": 1, "06": 2, "02": 3, "05": 4, "11": 9}


def _norm_id(raw: object) -> str:
    s = str(raw).strip().strip('"').strip("'").upper().replace(".", "-")
    m = re.search(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}(?:-[0-9]{2}[A-Z]?)?)", s)
    return m.group(1) if m else s


def _sample_type_rank(barcode: str) -> Tuple[int, str]:
    parts = barcode.split("-")
    stype = parts[3][:2] if len(parts) >= 4 else "99"
    return _SAMPLE_TYPE_PRIORITY.get(stype, 5), barcode


def _find_extracted_file(c_dir: Path, name: str) -> Path | None:
    direct = c_dir / name
    if direct.is_file() and direct.stat().st_size > 100:
        return direct
    for p in c_dir.rglob("*"):
        if p.is_file() and p.stem.lower() == name.lower() and p.stat().st_size > 100:
            return p
    return None


def download_and_extract_cohort(cohort: str, data_dir: Path) -> Path:
    """Download cohort ZIP via resilient curl resume and extract exp/methy/mirna/survival."""
    c_dir = data_dir / cohort
    c_dir.mkdir(parents=True, exist_ok=True)
    mods = ["exp", "methy", "mirna", "survival"]
    if all(_find_extracted_file(c_dir, m) is not None for m in mods):
        return c_dir

    zip_name = COHORT_ARCHIVES[cohort]
    url = f"{BASE_URL}/{zip_name}"
    zip_path = data_dir / zip_name

    need_download = True
    if zip_path.exists() and zip_path.stat().st_size > 1024:
        try:
            with zipfile.ZipFile(zip_path) as zf:
                if zf.testzip() is None:
                    need_download = False
        except Exception:
            need_download = True

    if need_download:
        try:
            subprocess.run(
                [
                    "curl",
                    "-k",
                    "-fL",
                    "--retry",
                    "6",
                    "--retry-delay",
                    "2",
                    "--retry-all-errors",
                    "-C",
                    "-",
                    "-o",
                    str(zip_path),
                    url,
                ],
                check=True,
            )
        except subprocess.CalledProcessError:
            if zip_path.exists():
                zip_path.unlink()
            subprocess.run(
                [
                    "curl",
                    "-k",
                    "-fL",
                    "--retry",
                    "6",
                    "--retry-delay",
                    "2",
                    "--retry-all-errors",
                    "-o",
                    str(zip_path),
                    url,
                ],
                check=True,
            )

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(c_dir)

    for m in mods:
        found = _find_extracted_file(c_dir, m)
        target = c_dir / m
        if found is not None and found != target:
            shutil.copy2(found, target)
    return c_dir


def _read_r_omic_matrix(path: Path) -> pd.DataFrame:
    """Parse R write.table matrix (samples as columns or rows) and standardize rows=samples."""
    df = pd.read_csv(path, sep=r"\s+")
    idx_hits = sum("TCGA" in str(i).upper() for i in df.index[:25])
    col_hits = sum("TCGA" in str(c).upper() for c in df.columns[:25])
    if col_hits > idx_hits:
        df = df.T
    df.index = [_norm_id(i) for i in df.index]
    df = df.apply(pd.to_numeric, errors="coerce")
    return df


def _dedup_by_priority(df: pd.DataFrame, key_len: int) -> pd.DataFrame:
    order = sorted(range(len(df)), key=lambda idx: _sample_type_rank(str(df.index[idx])))
    df_sorted = df.iloc[order].copy()
    df_sorted.index = [str(i)[:key_len] for i in df_sorted.index]
    return df_sorted[~pd.Index(df_sorted.index).duplicated(keep="first")]


def _read_survival_table(path: Path) -> pd.DataFrame:
    surv = pd.read_csv(path, sep=r"\s+")
    cols_lower = {str(c).strip().strip('"').lower(): c for c in surv.columns}
    id_col = next(
        (cols_lower[k] for k in ["patientid", "sampleid", "sample", "submitter_id"] if k in cols_lower),
        surv.columns[0],
    )
    t_col = next(
        (cols_lower[k] for k in ["survival", "os.time", "time", "days"] if k in cols_lower),
        surv.columns[1],
    )
    e_col = next(
        (cols_lower[k] for k in ["death", "os", "event", "status"] if k in cols_lower),
        surv.columns[2],
    )
    out = pd.DataFrame(
        {
            "barcode": [_norm_id(x) for x in surv[id_col]],
            "time": pd.to_numeric(surv[t_col], errors="coerce"),
            "event": pd.to_numeric(surv[e_col], errors="coerce"),
        }
    ).dropna()
    out = out[(out["time"] > 0) & (out["event"].isin([0, 1]))].copy()
    out = out.set_index("barcode")
    return out


def load_tcga_cohort(
    cohort: str,
    data_dir: Path,
    top_features: int = 2000,
) -> Tuple[List[np.ndarray], pd.DataFrame, List[str], Dict[str, object]]:
    """Download, align, log2-transform sequencing counts, and variance-filter one TCGA cohort."""
    c_dir = download_and_extract_cohort(cohort, data_dir)
    omics_raw: Dict[str, pd.DataFrame] = {}
    raw_shapes: Dict[str, int] = {}

    for mod in ["exp", "methy", "mirna"]:
        df = _read_r_omic_matrix(c_dir / mod)
        raw_shapes[mod] = int(df.shape[1])
        omics_raw[mod] = df

    surv_raw = _read_survival_table(c_dir / "survival")

    # Try 15-char sample barcode intersection first, then fall back to 12-char patient barcode
    om15 = {m: _dedup_by_priority(df, 15) for m, df in omics_raw.items()}
    sv15 = _dedup_by_priority(surv_raw, 15)
    common15 = sorted(set(om15["exp"].index) & set(om15["methy"].index) & set(om15["mirna"].index) & set(sv15.index))

    match_mode = "15-char"
    if len(common15) >= 40:
        # Collapse any dual aliquots of the same 12-char patient to prevent patient pseudoreplication
        om_aligned = {m: _dedup_by_priority(om15[m].loc[common15], 12) for m in om15}
        sv_aligned = _dedup_by_priority(sv15.loc[common15], 12)
        common = sorted(
            set(om_aligned["exp"].index)
            & set(om_aligned["methy"].index)
            & set(om_aligned["mirna"].index)
            & set(sv_aligned.index)
        )
    else:
        match_mode = "12-char"
        om_aligned = {m: _dedup_by_priority(df, 12) for m, df in omics_raw.items()}
        sv_aligned = _dedup_by_priority(surv_raw, 12)
        common = sorted(
            set(om_aligned["exp"].index)
            & set(om_aligned["methy"].index)
            & set(om_aligned["mirna"].index)
            & set(sv_aligned.index)
        )

    views: List[np.ndarray] = []
    meta: Dict[str, object] = {
        "cohort": cohort.upper(),
        "canonical_k": CANONICAL_K[cohort],
        "n_samples": len(common),
        "match_mode": match_mode,
        "common15_raw": len(common15),
    }

    surv_final = sv_aligned.loc[common].reset_index(drop=True)
    meta["n_events"] = int(surv_final["event"].sum())
    meta["event_rate"] = float(surv_final["event"].mean()) if len(surv_final) > 0 else 0.0

    for mod in ["exp", "methy", "mirna"]:
        sub = om_aligned[mod].loc[common].copy()
        sub = sub.dropna(axis=1, how="all")
        med = sub.median(axis=0)
        sub = sub.fillna(med).fillna(0.0)

        vals = sub.to_numpy(dtype=np.float64)
        vmin = float(np.nanmin(vals))
        vmax = float(np.nanmax(vals))
        log2_applied = False
        if mod in ("exp", "mirna") and vmin >= -1e-6 and vmax > 50.0:
            vals = np.log2(1.0 + np.clip(vals, 0.0, None))
            log2_applied = True

        var = vals.var(axis=0)
        keep_mask = var > 1e-10
        vals = vals[:, keep_mask]
        var = var[keep_mask]
        if vals.shape[1] > top_features:
            top_idx = np.argsort(var)[-top_features:]
            top_idx.sort()
            vals = vals[:, top_idx]

        views.append(vals.astype(np.float32))
        meta[f"raw_d_{mod}"] = raw_shapes[mod]
        meta[f"d_{mod}"] = int(vals.shape[1])
        meta[f"log2_{mod}"] = log2_applied

    return views, surv_final, common, meta

In [ ]:
# Cell 3: Download and preprocess all 10 TCGA cohorts (HTTPS + curl resume + R write.table + auto log2)
loaded_cohorts = {}
cohort_meta_rows = []
for cohort in COHORTS:
    vs, sv, ids, meta = load_tcga_cohort(cohort, DATA_DIR, top_features=2000)
    loaded_cohorts[cohort] = (vs, sv, ids, meta)
    cohort_meta_rows.append(meta)
    print(f"Loaded {cohort.upper():4s}: n={meta['n_samples']:3d}, events={meta['n_events']:3d}, match={meta['match_mode']}")

summary_df = pd.DataFrame(cohort_meta_rows)
summary_df.to_csv(OUT_DIR / "cohort_summary_v2.csv", index=False)
display(summary_df)

In [ ]:
# Cell 4: 10-TCGA Clean Benchmark (12 methods/ablations, 5000-permutation log-rank) + Partial Multi-Omics
import time

def benjamini_hochberg(pvals):
    p = np.asarray(pvals, dtype=np.float64)
    m = len(p)
    order = np.argsort(p)
    q = np.minimum.accumulate((p[order] * m / np.arange(1, m + 1))[::-1])[::-1].clip(0.0, 1.0)
    out = np.empty_like(q)
    out[order] = q
    return out

bench_rows = []
for cohort, (views, surv, ids, _) in loaded_cohorts.items():
    k_c = CANONICAL_K[cohort]
    runners = {
        "CL-VSSC (No-DZ PI)": lambda v, k: CLVSSC(k, dead_zone=0.0, seed=42).fit(v),
        "CL-VSSC (DZ-PI)": lambda v, k: CLVSSC(k, dead_zone=0.14, seed=42).fit(v),
        "CL-VSSC (P-only)": lambda v, k: CLVSSC(k, kappa_i=0.0, seed=42).fit(v),
        "CL-VSSC (w/o Feedback)": lambda v, k: CLVSSC(k, use_feedback=False, seed=42).fit(v),
        "OpenLoop_MAVS": lambda v, k: run_openloop_mavs(v, k, seed=42),
        "SNF": lambda v, k: run_snf(v, k, seed=42),
        "NEMO": lambda v, k: run_nemo(v, k, seed=42),
        "MCGC": lambda v, k: run_mcgc(v, k, seed=42),
        "Seurat-WNN": lambda v, k: run_wnn(v, k, seed=42),
        "Attention-MOAE": lambda v, k: run_attention_moae(v, k, epochs=100, seed=42),
        "Subtype-GAN": lambda v, k: run_subtype_gan(v, k, epochs=100, seed=42),
    }
    for m_name, fn in runners.items():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        out = fn(views, k_c)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        dt = time.time() - t0
        met = evaluate_clustering(out["labels"], survival_df=surv, embedding=out.get("F"), compute_perm=True, n_perm=5000, perm_seed=42)
        bench_rows.append({"cohort": cohort.upper(), "method": m_name, "neg_log10_perm_p": met["neg_log10_perm_p"], "perm_p": met["perm_p"], "silhouette": met.get("silhouette", np.nan), "runtime_sec": round(dt, 3)})

bench_df = pd.DataFrame(bench_rows)
for m_name, grp in bench_df.groupby("method"):
    bench_df.loc[grp.index, "fdr_q_perm"] = benjamini_hochberg(grp["perm_p"].to_numpy())
bench_df.to_csv(OUT_DIR / "tcga_benchmark_v2.csv", index=False)
display(bench_df.groupby("method")[["neg_log10_perm_p", "silhouette", "runtime_sec"]].agg(["mean", "median"]).round(3))

In [ ]:
# Cell 5: Large-Scale Single-Cell CITE-seq Demo on NCBI GEO GSE194122 (CL-VSSC-Anchor)
import gzip, shutil, anndata as ad, scipy.sparse as sp

BMMC_GZ = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE194nnn/GSE194122/suppl/GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad.gz"
gz_file = SC_DIR / "bmmc_cite.h5ad.gz"
h5_file = SC_DIR / "bmmc_cite.h5ad"
if not h5_file.exists():
    subprocess.run(["curl", "-k", "-fL", "--retry", "5", "-o", str(gz_file), BMMC_GZ], check=True)
    with gzip.open(gz_file, "rb") as fi, open(h5_file, "wb") as fo:
        shutil.copyfileobj(fi, fo)

ad_bmmc = ad.read_h5ad(h5_file)
raw_lbl = ad_bmmc.obs["cell_type"].astype(str)
vc = raw_lbl.value_counts()
keep = np.flatnonzero(raw_lbl.isin(set(vc[vc >= 15].index)).to_numpy())
rng = np.random.default_rng(42)
sub_idx = np.sort(rng.choice(keep, size=min(10000, len(keep)), replace=False))
_, y_int = np.unique(raw_lbl.iloc[sub_idx].to_numpy(), return_inverse=True)

feat = ad_bmmc.var["feature_types"].astype(str)
X_src = ad_bmmc.layers["counts"]
X_rna = X_src[sub_idx][:, (feat == "GEX").to_numpy()].tocsr().astype(np.float32)
inv_lib = 1e4 / np.maximum(np.asarray(X_rna.sum(axis=1)).ravel(), 1.0)
X_rna = X_rna.multiply(inv_lib[:, None]).tocsr()
X_rna.data = np.log1p(X_rna.data)
var_g = np.asarray(X_rna.power(2).mean(axis=0)).ravel() - np.asarray(X_rna.mean(axis=0)).ravel() ** 2
X_rna_dense = X_rna[:, np.sort(np.argsort(var_g)[-2000:])].toarray().astype(np.float32)
adt_raw = X_src[sub_idx][:, (feat == "ADT").to_numpy()].toarray().astype(np.float32)
adt_log = np.log1p(adt_raw)
X_adt = adt_log - adt_log.mean(axis=1, keepdims=True)

res_anc = CLVSSCAnchor(n_clusters=int(y_int.max() + 1), n_anchors=384, anchor_k=20, dead_zone=0.0, seed=42).fit([X_rna_dense, X_adt])
met_anc = evaluate_clustering(res_anc["labels"], true_labels=y_int, embedding=res_anc["F"])
print("CL-VSSC-Anchor on BMMC-10k:", {k: round(v, 4) for k, v in met_anc.items()}, "| weights:", np.round(res_anc["final_weights"], 3))

In [ ]:
# Cell 6: v3 Upgrades (Null-SNR PI, Double-Centered Anchor-Local PI, & mu_max Sensitivity)
OUT_DIR_V3 = Path("./results_v3")
OUT_DIR_V3.mkdir(parents=True, exist_ok=True)

# 1. Null-SNR PI clean vs corrupted mRNA on BIC
v_bic, surv_bic, _, _ = loaded_cohorts["bic"]
res_bic_null = CLVSSC(CANONICAL_K["bic"], use_null_snr=True, seed=42).fit(v_bic)
save_run_weights("bic_null_snr_clean", res_bic_null)
v_bic_corr = corrupt_view(v_bic, view_idx=0, noise_std=4.0, dropout_rate=0.60, seed=40)
res_bic_null_f = CLVSSC(CANONICAL_K["bic"], use_null_snr=True, seed=42).fit(v_bic_corr)
save_run_weights("bic_null_snr_fault", res_bic_null_f)
print("BIC Null-SNR clean weights:", np.round(res_bic_null["final_weights"], 4), "| mRNA fault w_corr:", round(float(res_bic_null_f["final_weights"][0]), 4))

# 2. Double-Centered Anchor-Local + Null PI on BMMC-10k
res_loc = CLVSSCAnchor(n_clusters=int(y_int.max() + 1), n_anchors=384, anchor_k=20, use_local_pi=True, use_null_snr=True, seed=42).fit([X_rna_dense, X_adt])
save_run_weights("bmmc10k_anchor_local_null_pi", res_loc)
met_loc = evaluate_clustering(res_loc["labels"], true_labels=y_int, embedding=res_loc["F"])
print("BMMC-10k Anchor (Local+Null):", {k: round(v, 4) for k, v in met_loc.items()})

# 3. Spectral-gap gain mu_max sensitivity check on BIC
for mu_m in [0.0, 0.12, 0.24]:
    r_mu = CLVSSC(CANONICAL_K["bic"], mu_max=mu_m, use_spectral_control=(mu_m > 0.0), seed=42).fit(v_bic)
    save_run_weights(f"bic_mu_{mu_m:.2f}", r_mu)
    m_mu = evaluate_clustering(r_mu["labels"], survival_df=surv_bic, embedding=r_mu["F"], compute_perm=False)
    print(f"BIC mu_max={mu_m:.2f}: gap_final={r_mu['spectral_gap_history'][-1]:.4f}, sil={m_mu.get('silhouette', np.nan):.4f}")


In [ ]:
# Cell 7: Package & Auto-Upload Categorized Results, Weights, and Run Manifest to Google Drive
run_stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Sync individual result files & weights into categorized Drive folders
sync_to_drive(OUT_DIR, "results_v2")
sync_to_drive(OUT_DIR_V3, "results_v3")
sync_to_drive(WEIGHTS_DIR, "weights")

# 2. Create timestamped zip bundles and sync to 05_Bundles_Zip/
zip_v2 = Path(shutil.make_archive(f"./cl_vssc_results_v2_{run_stamp}", "zip", OUT_DIR))
zip_v3 = Path(shutil.make_archive(f"./cl_vssc_results_v3_{run_stamp}", "zip", OUT_DIR_V3))
zip_w = Path(shutil.make_archive(f"./cl_vssc_weights_{run_stamp}", "zip", WEIGHTS_DIR))
for zf in [zip_v2, zip_v3, zip_w]:
    sync_to_drive(zf, "bundles")

# 3. Save run manifest & GPU environment to 06_Logs_Env/
manifest = {
    "timestamp": run_stamp,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "results_v2_files": sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()),
    "results_v3_files": sorted(p.name for p in OUT_DIR_V3.iterdir() if p.is_file()),
    "weight_files": sorted(p.name for p in WEIGHTS_DIR.iterdir() if p.is_file()),
}
manifest_path = Path(f"./run_manifest_{run_stamp}.json")
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
sync_to_drive(manifest_path, "logs")
print("All categorized outputs, weights, bundles, and manifest synced to:", DRIVE_ROOT)
